In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import warnings
import sys
sys.path.append('../')
from src import *

seed = 42
np.random.seed(seed)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_parquet("../data/all_raman_spectra.parquet")

In [4]:
# df = df.iloc[:1000000]

In [5]:
X, y = df[['X', 'Y', 'Wave', 'Intensity', 'group', 'center', 'brain']], df['label']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
del X, y

In [7]:
model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=42,
    task_type='GPU',
    verbose=200,
    early_stopping_rounds=100,
)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126291375 entries, 0 to 126291374
Data columns (total 9 columns):
 #   Column     Dtype   
---  ------     -----   
 0   X          float32 
 1   Y          float32 
 2   Wave       float32 
 3   Intensity  float32 
 4   label      category
 5   group      category
 6   center     category
 7   brain      category
 8   place      category
dtypes: category(5), float32(4)
memory usage: 2.5 GB


In [8]:
from sklearn.metrics import f1_score
import joblib


model.fit(X_train, y_train, use_best_model=True, plot=True)
# joblib.dump(model, 'model-catboost.joblib')

y_pred = model.predict(X_test)

score = f1_score(y_test, y_pred)

CatBoostError: features data: pandas.DataFrame column 'group' has dtype 'category' but is not in  cat_features list